In [1]:
import torch
import einops

In [2]:
def vector_gather(vectors, indices):
    """
    Gathers (batched) vectors according to indices.
    Arguments:
        vectors: Tensor[N, L, D]
        indices: Tensor[N, K] or Tensor[N]
    Returns:
        Tensor[N, K, D] or Tensor[N, D]
    """
    N, L, D = vectors.shape
    squeeze = False
    if indices.ndim == 1:
        squeeze = True
        indices = indices.unsqueeze(-1)
    N2, K = indices.shape
    assert N == N2
    indices = einops.repeat(indices, "N K -> N K D", D=D)
    out = torch.gather(vectors, dim=1, index=indices)
    if squeeze:
        out = out.squeeze(1)
    return out

In [3]:
def dag_loss(targets, transition_matrix, emission_probs, bos_idx=0):
    batch_size, m = targets.shape
    _, l, vocab_size = emission_probs.shape
    dp = torch.zeros((batch_size, m, l))
    bos_emissions = emission_probs[:, 0, bos_idx]
    dp[:, 0, 0] = bos_emissions
    # dp is almost setup correctly, just need to replace every 0 with -inf
    dp[dp == 0] = -float('inf')
    # assumes that transition_matrix and emission_probs are already in log space
    # also we need to tranpose emission_probs so it is vocab_size x l
    # so the vector gather works
    emission_probs = emission_probs.transpose(1, 2)
    for i in range(1, m):
        dp[:, i, :] = vector_gather(emission_probs, targets[:, i]) + (torch.logsumexp(dp[:, i-1, :].unsqueeze(1).transpose(1, 2) + transition_matrix, dim=1))
    return dp

In [4]:
def fix_probs(logprobs, mask):
    # assumes probs is already in log space
    # and is a square matrix
    # updates probs so that the sum of each row is 1
    # and any available probability mass is 
    # distributed evenly among the non-masked entries
    batch_size, l, _ = logprobs.shape
    logprobs = logprobs.masked_fill(mask == 1, float('-inf'))
    probsmatrix = torch.exp(logprobs)
    remaining = torch.sum(probsmatrix, dim=2)
    remaining = 1 - remaining
    probnonzero = torch.sum(mask == 0, dim=-1)
    remaining = remaining / probnonzero
    probsmatrix = probsmatrix + remaining.unsqueeze(2)
    probsmatrix = probsmatrix.masked_fill(mask == 1, 0)
    logprobs = torch.log(probsmatrix)
    return logprobs

In [5]:
vocab_size = 5
example_1_len = 3
factor = 2
num_vertices = example_1_len * factor

In [6]:
target_tokens = torch.randint(0, vocab_size, (1, example_1_len))

In [7]:
emissions_prob  = torch.rand((1, num_vertices, vocab_size))
transition_matrix = torch.rand((1, num_vertices, num_vertices))

In [8]:
# ensure transition_matrix contains valid probability distributions
transition_matrix = torch.nn.functional.softmax(transition_matrix, dim=2)

In [9]:
transition_matrix

tensor([[[0.1821, 0.1367, 0.1223, 0.1479, 0.1300, 0.2810],
         [0.1841, 0.1382, 0.1657, 0.1389, 0.2073, 0.1658],
         [0.1497, 0.1666, 0.1606, 0.2356, 0.1618, 0.1258],
         [0.1212, 0.2409, 0.1070, 0.1604, 0.1160, 0.2545],
         [0.1499, 0.1450, 0.1606, 0.1739, 0.1548, 0.2157],
         [0.1251, 0.1743, 0.2209, 0.1055, 0.2031, 0.1712]]])

In [10]:
emissions_prob = torch.nn.functional.softmax(emissions_prob, dim=2)

In [11]:
emissions_prob

tensor([[[0.1191, 0.1898, 0.2827, 0.1370, 0.2714],
         [0.2129, 0.1401, 0.2343, 0.1964, 0.2164],
         [0.1776, 0.2105, 0.2516, 0.1153, 0.2451],
         [0.1442, 0.2411, 0.2517, 0.2174, 0.1455],
         [0.1011, 0.2562, 0.2466, 0.1324, 0.2636],
         [0.1460, 0.2549, 0.2319, 0.2283, 0.1389]]])

In [12]:
mask = torch.tril(torch.ones((1, num_vertices, num_vertices)))

In [13]:
transition_matrix = torch.log(transition_matrix)

In [14]:
transition_matrix = fix_probs(transition_matrix, mask)

In [15]:
emissions_prob = torch.log(emissions_prob)

In [16]:
dp = dag_loss(target_tokens, transition_matrix, emissions_prob)

In [17]:
dp[0][-1][-1]

tensor(-6.0104)

In [18]:
example_2_len = 4
num_vertices_2 = example_2_len * factor

In [19]:
transition_matrix_2 = torch.rand((1, num_vertices_2, num_vertices_2))
emit_probs_2 = torch.rand((1, num_vertices_2, vocab_size))
target_tokens_2 = torch.randint(0, vocab_size, (1, example_2_len))

In [20]:
transition_matrix_2 = torch.nn.functional.softmax(transition_matrix_2, dim=2)
emit_probs_2 = torch.nn.functional.softmax(emit_probs_2, dim=2)

In [21]:
mask2 = torch.tril(torch.ones((1, num_vertices_2, num_vertices_2)))
transition_matrix_2 = torch.log(transition_matrix_2)
transition_matrix_2 = fix_probs(transition_matrix_2, mask2)

In [22]:
emit_probs_2 = torch.log(emit_probs_2)

In [23]:
dp2 = dag_loss(target_tokens_2, transition_matrix_2, emit_probs_2)

In [24]:
dp2[0][-1][-1]

tensor(-7.5569)

In [25]:
padded_length = 10
assert padded_length >= example_1_len and padded_length >= example_2_len
num_padded_vertices = padded_length * factor
assert num_padded_vertices >= num_vertices and num_padded_vertices >= num_vertices_2

In [26]:
# pad the emissions matrices so they are both 1 x padded_length x vocab_size
emissions_prob_padded = torch.zeros((1, num_padded_vertices, vocab_size))
emissions_prob_padded[:, :num_vertices, :] = emissions_prob
emissions_prob_padded[:, num_vertices:, :] = float('-inf')

emissions_prob_padded_2 = torch.zeros((1, num_padded_vertices, vocab_size))
emissions_prob_padded_2[:, :num_vertices_2, :] = emit_probs_2
emissions_prob_padded_2[:, num_vertices_2:, :] = float('-inf')

# pad the transition matrices so they are both 1 x num_padded_vertices x num_padded_vertices
transition_matrix_padded = torch.zeros((1, num_padded_vertices, num_padded_vertices))
transition_matrix_padded[transition_matrix_padded == 0] = float('-inf')
transition_matrix_padded[:, :num_vertices, :num_vertices] = transition_matrix

transition_matrix_padded_2 = torch.zeros((1, num_padded_vertices, num_padded_vertices))
transition_matrix_padded_2[transition_matrix_padded_2 == 0] = float('-inf')
transition_matrix_padded_2[:, :num_vertices_2, :num_vertices_2] = transition_matrix_2

In [27]:
transition_matrix

tensor([[[   -inf, -1.7536, -1.8409, -1.6910, -1.7932, -1.1475],
         [   -inf,    -inf, -1.4011, -1.5166, -1.2453, -1.4010],
         [   -inf,    -inf,    -inf, -0.9301, -1.1370, -1.2562],
         [   -inf,    -inf,    -inf,    -inf, -0.8422, -0.5635],
         [   -inf,    -inf,    -inf,    -inf,    -inf,  0.0000],
         [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf]]])

In [28]:
transition_matrix_padded

tensor([[[   -inf, -1.7536, -1.8409, -1.6910, -1.7932, -1.1475,    -inf,
             -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
             -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
         [   -inf,    -inf, -1.4011, -1.5166, -1.2453, -1.4010,    -inf,
             -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
             -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
         [   -inf,    -inf,    -inf, -0.9301, -1.1370, -1.2562,    -inf,
             -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
             -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
         [   -inf,    -inf,    -inf,    -inf, -0.8422, -0.5635,    -inf,
             -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
             -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
         [   -inf,    -inf,    -inf,    -inf,    -inf,  0.0000,    -inf,
             -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
          

In [29]:
# create batched version 
batched_emissions = torch.zeros((2, num_padded_vertices, vocab_size))
batched_emissions[0] = emissions_prob_padded
batched_emissions[1] = emissions_prob_padded_2

batched_transitions = torch.zeros((2, num_padded_vertices, num_padded_vertices))
batched_transitions[0] = transition_matrix_padded
batched_transitions[1] = transition_matrix_padded_2

In [30]:
target_lens = torch.tensor([example_1_len, example_2_len])

In [31]:
target_lens

tensor([3, 4])

In [32]:
targets = torch.nested.nested_tensor([target_tokens.squeeze(0), target_tokens_2.squeeze(0)])

/tmp/ipykernel_35449/3810897822.py:1: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  targets = torch.nested.nested_tensor([target_tokens.squeeze(0), target_tokens_2.squeeze(0)])


In [33]:
targets

nested_tensor([
  tensor([1, 2, 2]),
  tensor([0, 4, 3, 3])
])

In [34]:
targets = torch.nested.to_padded_tensor(targets, 1, (2, padded_length))

In [35]:
batch_dp = dag_loss(targets, batched_transitions, batched_emissions)

In [36]:
batch_dp.shape

torch.Size([2, 10, 20])

In [37]:
dp2

tensor([[[ -1.2193,     -inf,     -inf,     -inf,     -inf,     -inf,     -inf,
              -inf],
         [    -inf,  -4.6068,  -5.4918,  -4.5069,  -4.9053,  -5.0549,  -4.4912,
           -5.1892],
         [    -inf,     -inf,  -8.0965,  -7.7014,  -6.7571,  -6.3966,  -5.8789,
           -5.8719],
         [    -inf,     -inf,     -inf, -11.6739, -10.0340,  -8.9762,  -7.9010,
           -7.5569]]])

In [38]:
batch_dp[1][example_2_len -1][example_2_len * factor - 1]

tensor(-7.5569)

In [39]:
dp_vectors = vector_gather(batch_dp, target_lens - 1)

In [40]:
values = torch.gather(dp_vectors, dim=1, index=(target_lens * factor - 1).unsqueeze(-1))

In [41]:
values

tensor([[-6.0104],
        [-7.5569]])

In [42]:
# values[0][0] should be dp target value, and values[1][0] should be dp2 target value
assert torch.allclose(values[0][0], dp[0][-1][-1])
assert torch.allclose(values[1][0], dp2[0][-1][-1])

In [43]:
# # pad the emissions matrices so they are both 1 x padded_length x vocab_size
# emissions_prob_padded = torch.zeros((1, num_padded_vertices, vocab_size))
# emissions_prob_padded[:, :num_vertices, :] = emissions_prob
# emissions_prob_padded[:, num_vertices:, :] = float('-inf')

# emissions_prob_padded_2 = torch.zeros((1, num_padded_vertices, vocab_size))
# emissions_prob_padded_2[:, :num_vertices_2, :] = emit_probs_2
# emissions_prob_padded_2[:, num_vertices_2:, :] = float('-inf')

# # pad the transition matrices so they are both 1 x num_padded_vertices x num_padded_vertices
# transition_matrix_padded = torch.zeros((1, num_padded_vertices, num_padded_vertices))
# transition_matrix_padded[transition_matrix_padded == 0] = float('-inf')
# transition_matrix_padded[:, :num_vertices, :num_vertices] = transition_matrix

# transition_matrix_padded_2 = torch.zeros((1, num_padded_vertices, num_padded_vertices))
# transition_matrix_padded_2[transition_matrix_padded_2 == 0] = float('-inf')
# transition_matrix_padded_2[:, :num_vertices_2, :num_vertices_2] = transition_matrix_2

# we'll do something similar to the above, but instead of -inf, we'll use random values
emissions_prob_padded = torch.rand((1, num_padded_vertices, vocab_size))
emissions_prob_padded[:, :num_vertices, :] = emissions_prob

emissions_prob_padded_2 = torch.rand((1, num_padded_vertices, vocab_size))
emissions_prob_padded_2[:, :num_vertices_2, :] = emit_probs_2

transition_matrix_padded = torch.rand((1, num_padded_vertices, num_padded_vertices))
transition_matrix_padded[:, :num_vertices, :num_vertices] = transition_matrix

transition_matrix_padded_2 = torch.rand((1, num_padded_vertices, num_padded_vertices))
transition_matrix_padded_2[:, :num_vertices_2, :num_vertices_2] = transition_matrix_2

In [44]:
batched_emissions2 = torch.zeros((2, num_padded_vertices, vocab_size))
batched_emissions2[0] = emissions_prob_padded
batched_emissions2[1] = emissions_prob_padded_2

batched_transitions2 = torch.zeros((2, num_padded_vertices, num_padded_vertices))
batched_transitions2[0] = transition_matrix_padded
batched_transitions2[1] = transition_matrix_padded_2

In [45]:
target_lens

tensor([3, 4])

In [46]:
target_lens_mask = torch.arange(padded_length * factor).repeat(len(target_lens), 1) < (target_lens * 2).unsqueeze(-1)

In [47]:
# test describes which rows of batched_emissions to mask, if it is 0, mask it
batched_emissions2[target_lens_mask == 0] = float('-inf')

In [48]:
batched_transitions2.transpose(1, 2)[target_lens_mask == 0] = float('-inf')

In [49]:
batch_dp2 = dag_loss(targets, batched_transitions2, batched_emissions2)

In [50]:
dp_vectors2 = vector_gather(batch_dp2, target_lens - 1)

In [51]:
dp_vectors2

tensor([[    -inf,     -inf,  -8.1138,  -7.2102,  -6.6405,  -6.0104,     -inf,
             -inf,     -inf,     -inf,     -inf,     -inf,     -inf,     -inf,
             -inf,     -inf,     -inf,     -inf,     -inf,     -inf],
        [    -inf,     -inf,     -inf, -11.6739, -10.0340,  -8.9762,  -7.9010,
          -7.5569,     -inf,     -inf,     -inf,     -inf,     -inf,     -inf,
             -inf,     -inf,     -inf,     -inf,     -inf,     -inf]])

In [52]:
values2 = torch.gather(dp_vectors2, dim=1, index=(target_lens * factor - 1).unsqueeze(-1))

In [53]:
values2

tensor([[-6.0104],
        [-7.5569]])

In [54]:
values

tensor([[-6.0104],
        [-7.5569]])